In [0]:
jdbc_url = "jdbc:postgresql://0.tcp.in.ngrok.io:26324/demo"

username = "postgres"
password = "root"

driver = "org.postgresql.Driver"

# Finding Last_Updated_AT

In [0]:
last_watermark = (
    spark.sql("""
        SELECT MAX(updated_at) AS max_updated_at
        FROM migration.raw.customers
    """)
    .collect()[0]["max_updated_at"]
)

print("Last watermark:", last_watermark)

# Incremental Query

In [0]:
incremental_query = f"""
SELECT *
FROM public.customers
WHERE updated_at > '{last_watermark}'
"""

In [0]:
incremental_customers_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", incremental_query)
    .option("user", username)
    .option("password", password)
    .option("driver", driver)
    .load()
)

display(incremental_customers_df)

In [0]:
(
    incremental_customers_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("migration.raw.customers")
)

In [0]:
%sql
SELECT *
FROM migration.raw.customers
ORDER BY customer_id, updated_at;

Creating Batch_ID

In [0]:
import uuid 

batch_id = str(uuid.uuid4())
print("Batch ID : ", batch_id)

# Adding MetaData in incremental batch

In [0]:
from pyspark.sql.functions import current_timestamp, lit
